In [1]:
import polars as pl
from polars import col
from investment_strategy.data.cleaner import *
from investment_strategy.signals.signal_construction import *
from investment_strategy.signals.signal_ranking import *
from investment_strategy.portfolio.weighting import *
from investment_strategy.portfolio.rebalancing import *
from investment_strategy.portfolio.valuation import *
from investment_strategy.analytics.return_metrics import *
from investment_strategy.analytics.risk_metrics import *
from investment_strategy.config.backtest_config import *
from investment_strategy.analytics.turnover_metrics import *
from investment_strategy.analytics.benchmark_metrics import *

market_data = pl.read_parquet("../data/raw/sp500_market_data.parquet")
market_data

date,ticker,open,high,low,close,volume
date,str,f64,f64,f64,f64,i64
2018-12-31,"""A""",62.795802,63.874904,62.795802,63.855968,1572100
2018-12-31,"""AAPL""",37.613949,37.810881,37.127551,37.42651,140014000
2018-12-31,"""ABBV""",66.040205,67.042343,65.773452,66.465576,5722100
2018-12-31,"""ABNB""",null,null,null,null,null
2018-12-31,"""ABT""",62.168729,63.202416,62.090555,62.828899,6094300
…,…,…,…,…,…,…
2026-05-29,"""XYZ""",74.970001,76.660004,74.195,75.720001,7380400
2026-05-29,"""YUM""",149.229996,150.179993,147.339996,147.949997,3992700
2026-05-29,"""ZBH""",81.952229,82.979498,81.363796,82.111809,3216600


In [2]:
benchmark_data = pl.read_parquet("../data/raw/benchmark_market_data.parquet")
benchmark_data

date,ticker,open,high,low,close,volume
date,str,f64,f64,f64,f64,i64
2018-12-31,"""^DJI""",23153.939453,23333.179688,23118.300781,23327.460938,288830000
2018-12-31,"""^GSPC""",2498.939941,2509.23999,2482.820068,2506.850098,3461920000
2018-12-31,"""^IXIC""",6649.52002,6659.959961,6570.060059,6635.279785,2109320000
2018-12-31,"""^NDX""",6354.850098,6365.390137,6273.939941,6329.970215,2109320000
2018-12-31,"""^RUT""",1338.52002,1348.560059,1329.170044,1348.560059,3461920000
…,…,…,…,…,…,…
2026-05-29,"""^DJI""",50773.910156,51094.179688,50698.269531,51032.460938,894700000
2026-05-29,"""^GSPC""",7579.330078,7599.379883,7563.549805,7580.060059,7858290000
2026-05-29,"""^IXIC""",26960.839844,27094.800781,26859.269531,26972.619141,11906000000


# Signal Construction

In [3]:
market_data = fill_OHLCV_missing_values(market_data)
close_price = market_data.select(
    col("date"),
    col("ticker"),
    col("close")
)

In [4]:
end_date = get_backtest_end_date(
    backtest_start_date=BACKTEST_START_DATE,
    backtest_period=BACKTEST_PERIOD,
    backtest_period_unit=BACKTEST_PERIOD_UNIT,
)
end_date

datetime.date(2024, 12, 31)

In [5]:
trading_calendar = get_trading_calendar(cleaned_close_prices_dataset=close_price)
trading_calendar

date
date
2018-12-31
2019-01-02
2019-01-03
2019-01-04
2019-01-07
…
2026-05-22
2026-05-26
2026-05-27


In [6]:
date_mapping_df = create_date_mapping(
    trading_calendar=trading_calendar,
    rebalance_frequency=REBALANCE_FREQUENCY,
    rebalance_freq_unit=REBALANCE_FREQ_UNIT,
    backtest_start_date=BACKTEST_START_DATE,
    backtest_end_date=end_date,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
date_mapping_df

lookback_date,lag_base_date,signal_date,rebalance_date
date,date,date,date
2021-06-30,2021-11-30,2021-12-30,2021-12-31
2021-08-25,2022-01-25,2022-02-25,2022-02-28
2021-10-29,2022-03-29,2022-04-29,2022-05-02
2021-12-29,2022-05-27,2022-06-29,2022-06-30
2022-02-28,2022-07-29,2022-08-30,2022-08-31
…,…,…,…
2023-10-27,2024-03-28,2024-04-29,2024-04-30
2023-12-28,2024-05-28,2024-06-28,2024-07-01
2024-02-29,2024-07-30,2024-08-30,2024-09-03


In [7]:
price_date_df = get_prices_for_date_mapping(
    cleaned_close_prices_dataset=close_price,
    cleaned_stock_OHLCV=market_data,
    date_mapping_df=date_mapping_df,
)
price_date_df

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open
date,str,f64,date,date,date,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822
…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847


In [8]:
past_returns = calculate_momentum(
    factor_reference_table=price_date_df,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
past_returns

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo
date,str,f64,date,date,date,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214
…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519


In [9]:
past_std = calculate_past_returns_std(
    cleaned_close_prices_dataset=close_price,
    factor_reference_table=past_returns,
    date_mapping_df=date_mapping_df,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
past_std

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo
date,str,f64,date,date,date,f64,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352,0.01249
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494,0.013089
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393,0.012661
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681,0.027615
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214,0.010467
…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081,0.027146
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154,0.010647
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519,0.015842


In [10]:
risk_adjusted_table = get_risk_adjusted_return(
    factor_reference_with_momentum_and_vol=past_std,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
risk_adjusted_table

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352,0.01249,1.883082
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494,0.013089,16.081247
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393,0.012661,3.743267
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681,0.027615,4.587432
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214,0.010467,8.905733
…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081,0.027146,13.743568
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154,0.010647,5.086498
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519,0.015842,2.221282


# Signal Ranking

In [11]:
signal_ranked = rank_signal(
    signal_df=risk_adjusted_table,
    signal_col=f"risk_adjusted_momentum_{LOOKBACK_PERIOD}{LOOKBACK_PERIOD_UNIT}",
)
signal_ranked

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352,0.01249,1.883082,255
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494,0.013089,16.081247,42
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393,0.012661,3.743267,213
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681,0.027615,4.587432,195
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214,0.010467,8.905733,111
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081,0.027146,13.743568,191
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154,0.010647,5.086498,341
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519,0.015842,2.221282,376


In [12]:
filtered_ticker = filter_top_ranked(
    ranked_signal_df=signal_ranked,
    rank_col=f"risk_adjusted_momentum_{LOOKBACK_PERIOD}{LOOKBACK_PERIOD_UNIT} rank",
    top_n=TOP_N,
)
filtered_ticker

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32
2021-12-30,"""ALB""",221.557343,2021-06-30,2021-11-30,2021-12-31,158.740082,251.533096,221.235862,0.584559,0.02389,24.468939,10
2021-12-30,"""AMD""",145.149994,2021-06-30,2021-11-30,2021-12-31,93.93,158.369995,146.160004,0.686043,0.025971,26.415665,8
2021-12-30,"""BLDR""",84.050003,2021-06-30,2021-11-30,2021-12-31,42.66,69.440002,84.290001,0.627754,0.020418,30.7454,3
2021-12-30,"""COST""",535.266479,2021-06-30,2021-11-30,2021-12-31,374.264069,511.982544,534.791945,0.367971,0.010224,35.990245,2
2021-12-30,"""DDOG""",178.929993,2021-06-30,2021-11-30,2021-12-31,104.080002,178.289993,179.190002,0.713009,0.026257,27.15499,6
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""NI""",35.226597,2024-06-28,2024-11-29,2024-12-31,27.213289,36.560795,35.274594,0.34349,0.009243,37.160354,5
2024-12-30,"""PLTR""",77.18,2024-06-28,2024-11-29,2024-12-31,25.33,67.080002,77.580002,1.648243,0.037635,43.795033,3
2024-12-30,"""TPL""",365.909271,2024-06-28,2024-11-29,2024-12-31,238.746994,528.161865,367.419189,1.212224,0.026436,45.855291,2


In [13]:
sorted_ranking = sort_rankings(
    filtered_signal_df=filtered_ticker,
    rank_col=f"risk_adjusted_momentum_{LOOKBACK_PERIOD}{LOOKBACK_PERIOD_UNIT} rank",
)
sorted_ranking

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32
2021-12-30,"""FDS""",461.847412,2021-06-30,2021-11-30,2021-12-31,318.493347,446.441071,461.847424,0.401728,0.010503,38.250071,1
2021-12-30,"""COST""",535.266479,2021-06-30,2021-11-30,2021-12-31,374.264069,511.982544,534.791945,0.367971,0.010224,35.990245,2
2021-12-30,"""BLDR""",84.050003,2021-06-30,2021-11-30,2021-12-31,42.66,69.440002,84.290001,0.627754,0.020418,30.7454,3
2021-12-30,"""VRSK""",221.089966,2021-06-30,2021-11-30,2021-12-31,168.90416,217.692719,220.925249,0.288854,0.009398,30.735613,4
2021-12-30,"""ODFL""",173.97641,2021-06-30,2021-11-30,2021-12-31,123.783798,173.439255,173.854314,0.401147,0.013245,30.28613,5
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""TRGP""",172.180573,2024-06-28,2024-11-29,2024-12-31,123.482193,197.887573,172.538972,0.60256,0.016254,37.070911,6
2024-12-30,"""ATO""",134.360123,2024-06-28,2024-11-29,2024-12-31,111.464882,146.342606,134.872702,0.312903,0.008631,36.252237,7
2024-12-30,"""AXON""",604.320007,2024-06-28,2024-11-29,2024-12-31,294.23999,646.960022,607.169983,1.198749,0.033564,35.715292,8


# Portfolio Weights

In [14]:
equal_weighted_portfolio = construct_portfolio_weights(
    filtered_signal_df=filtered_ticker, weighting_method="equal_weighted"
)
equal_weighted_portfolio

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank,portfolio_weight
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32,f64
2021-12-30,"""ALB""",221.557343,2021-06-30,2021-11-30,2021-12-31,158.740082,251.533096,221.235862,0.584559,0.02389,24.468939,10,0.1
2021-12-30,"""AMD""",145.149994,2021-06-30,2021-11-30,2021-12-31,93.93,158.369995,146.160004,0.686043,0.025971,26.415665,8,0.1
2021-12-30,"""BLDR""",84.050003,2021-06-30,2021-11-30,2021-12-31,42.66,69.440002,84.290001,0.627754,0.020418,30.7454,3,0.1
2021-12-30,"""COST""",535.266479,2021-06-30,2021-11-30,2021-12-31,374.264069,511.982544,534.791945,0.367971,0.010224,35.990245,2,0.1
2021-12-30,"""DDOG""",178.929993,2021-06-30,2021-11-30,2021-12-31,104.080002,178.289993,179.190002,0.713009,0.026257,27.15499,6,0.1
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""NI""",35.226597,2024-06-28,2024-11-29,2024-12-31,27.213289,36.560795,35.274594,0.34349,0.009243,37.160354,5,0.1
2024-12-30,"""PLTR""",77.18,2024-06-28,2024-11-29,2024-12-31,25.33,67.080002,77.580002,1.648243,0.037635,43.795033,3,0.1
2024-12-30,"""TPL""",365.909271,2024-06-28,2024-11-29,2024-12-31,238.746994,528.161865,367.419189,1.212224,0.026436,45.855291,2,0.1


In [15]:
rebalance_allocation_df = prepare_rebalance_allocation_df(
    weighted_portfolio_signal_df=equal_weighted_portfolio
)
rebalance_allocation_df

rebalance_date,ticker,rebalance_open,portfolio_weight
date,str,f64,f64
2021-12-31,"""ALB""",221.235862,0.1
2021-12-31,"""AMD""",146.160004,0.1
2021-12-31,"""BLDR""",84.290001,0.1
2021-12-31,"""COST""",534.791945,0.1
2021-12-31,"""DDOG""",179.190002,0.1
…,…,…,…
2024-12-31,"""NI""",35.274594,0.1
2024-12-31,"""PLTR""",77.580002,0.1
2024-12-31,"""TPL""",367.419189,0.1


In [16]:
rebalance_date = date_mapping_df.get_column("rebalance_date")
rebalance_date

rebalance_date
date
2021-12-31
2022-02-28
2022-05-02
2022-06-30
2022-08-31
…
2024-04-30
2024-07-01
2024-09-03


# Portfolio Construction

In [17]:
rebalance_summary = run_rebalance_simulation(
    factor_reference_table=price_date_df,
    rebalance_allocation_df=rebalance_allocation_df,
    initial_capital=INITIAL_CAPITAL,
    rebalance_dates=rebalance_date,
    execution_cost_rate=EXECUTION_COST_RATE,
    commission_per_share=COMMISSION_PER_SHARE
)
rebalance_summary["rebalance_level_table"]

rebalance_date,pre_rebalance_portfolio_value,post_rebalance_portfolio_value,cash_residual,transaction_cost
date,f64,f64,f64,f64
2021-12-31,1e6,998976.191123,1137.314027,1023.808877
2022-02-28,856221.430925,854411.858909,-476.039922,1809.572016
2022-05-02,927211.597011,925607.882371,-63.733927,1603.71464
2022-06-30,880665.677317,879133.012236,-275.40509,1532.665081
2022-08-31,999423.805348,997711.873931,127.361094,1711.931417
…,…,…,…,…
2024-04-30,1.8367e6,1.8332e6,-906.557785,3454.984999
2024-07-01,1.9034e6,1.9002e6,-777.462812,3225.969912
2024-09-03,1.9190e6,1.9149e6,-1450.29031,4106.055004


In [18]:
rebalance_summary["position_level_table"]

rebalance_date,ticker,shares
date,str,i64
2021-12-31,"""ALB""",451
2021-12-31,"""AMD""",683
2021-12-31,"""BLDR""",1185
2021-12-31,"""COST""",186
2021-12-31,"""DDOG""",557
…,…,…
2024-12-31,"""NI""",5513
2024-12-31,"""PLTR""",2507
2024-12-31,"""TPL""",529


In [19]:
rebalance_summary["trade_level_table"]

rebalance_date,ticker,shares_traded,rebalance_open,trade_value,execution_cost,commission,transaction_cost,cash_flows
date,str,i64,f64,f64,f64,f64,f64,f64
2021-12-31,"""ALB""",451,221.235862,99777.373681,99.777374,2.255,102.032374,-99879.406055
2021-12-31,"""AMD""",683,146.160004,99827.282501,99.827283,3.415,103.242283,-99930.524784
2021-12-31,"""BLDR""",1185,84.290001,99883.651085,99.883651,5.925,105.808651,-99989.459736
2021-12-31,"""COST""",186,534.791945,99471.301821,99.471302,0.93,100.401302,-99571.703122
2021-12-31,"""DDOG""",557,179.190002,99808.83136,99.808831,2.785,102.593831,-99911.425191
…,…,…,…,…,…,…,…,…
2024-12-31,"""NI""",5513,35.274594,194468.8379,194.468838,27.565,222.033838,-194690.871738
2024-12-31,"""PLTR""",2507,77.580002,194493.06459,194.493065,12.535,207.028065,-194700.092655
2024-12-31,"""TPL""",529,367.419189,194364.751229,194.364751,2.645,197.009751,-194561.76098


# Portfolio Valuation

In [20]:
rebalance_period_close_prices = get_backtest_period_close_prices(
    cleaned_close_prices_dataset=close_price,
    backtest_start_date=BACKTEST_START_DATE,
    backtest_end_date=end_date,
)
rebalance_period_close_prices

date,ticker,close
date,str,f64
2021-12-31,"""A""",154.27504
2021-12-31,"""AAPL""",173.599014
2021-12-31,"""ABBV""",114.065155
2021-12-31,"""ABNB""",166.490005
2021-12-31,"""ABT""",128.19101
…,…,…
2024-12-31,"""XYZ""",84.989998
2024-12-31,"""YUM""",130.370239
2024-12-31,"""ZBH""",104.037064


In [21]:
next_date_matched_rebalance_level_table = get_next_date_matched_rebalance_level_table(
    rebalance_level_table=rebalance_summary["rebalance_level_table"]
)
next_date_matched_rebalance_level_table

rebalance_date,pre_rebalance_portfolio_value,post_rebalance_portfolio_value,cash_residual,transaction_cost,next_rebalance_date
date,f64,f64,f64,f64,date
2021-12-31,1e6,998976.191123,1137.314027,1023.808877,2022-02-28
2022-02-28,856221.430925,854411.858909,-476.039922,1809.572016,2022-05-02
2022-05-02,927211.597011,925607.882371,-63.733927,1603.71464,2022-06-30
2022-06-30,880665.677317,879133.012236,-275.40509,1532.665081,2022-08-31
2022-08-31,999423.805348,997711.873931,127.361094,1711.931417,2022-10-31
…,…,…,…,…,…
2024-04-30,1.8367e6,1.8332e6,-906.557785,3454.984999,2024-07-01
2024-07-01,1.9034e6,1.9002e6,-777.462812,3225.969912,2024-09-03
2024-09-03,1.9190e6,1.9149e6,-1450.29031,4106.055004,2024-10-31


In [22]:
daily_position_value_table = get_daily_position_value_table(
    backtest_period_close_prices=rebalance_period_close_prices,
    next_date_matched_rebalance_level_table=next_date_matched_rebalance_level_table,
    position_level_table=rebalance_summary["position_level_table"],
)
daily_position_value_table

date,rebalance_date,ticker,shares,close,position_value
date,date,str,i64,f64,f64
2021-12-31,2021-12-31,"""ALB""",451,221.008972,99675.046448
2021-12-31,2021-12-31,"""AMD""",683,143.899994,98283.695831
2021-12-31,2021-12-31,"""BLDR""",1185,85.709999,101566.348915
2021-12-31,2021-12-31,"""COST""",186,538.864075,100228.717896
2021-12-31,2021-12-31,"""DDOG""",557,178.110001,99207.27034
…,…,…,…,…,…
2024-12-31,2024-12-31,"""NI""",5513,35.284191,194521.745708
2024-12-31,2024-12-31,"""PLTR""",2507,75.629997,189604.403114
2024-12-31,2024-12-31,"""TPL""",529,365.423492,193309.027496


In [23]:
daily_portfolio_table = get_daily_portfolio_table(
    next_date_matched_rebalance_level_table=next_date_matched_rebalance_level_table,
    daily_position_value_table=daily_position_value_table,
    initial_capital=INITIAL_CAPITAL
)
daily_portfolio_table

date,positions_value,cash_residual,portfolio_value,daily_return
date,f64,f64,f64,f64
2021-12-31,999775.987938,1137.314027,1.0009e6,0.000913
2022-01-03,984317.072136,1137.314027,985454.386163,-0.015445
2022-01-04,979994.58197,1137.314027,981131.895998,-0.004386
2022-01-05,939071.453529,1137.314027,940208.767557,-0.04171
2022-01-06,936567.684242,1137.314027,937704.99827,-0.002663
…,…,…,…,…
2024-12-24,1.9756e6,908.805957,1.9765e6,0.006458
2024-12-26,1.9718e6,908.805957,1.9728e6,-0.001911
2024-12-27,1.9560e6,908.805957,1.9569e6,-0.008037


# Analytics

## Return metrics

In [24]:
daily_portfolio_value_return_df = prepare_daily_portfolio_value_return_df(
    daily_portfolio_table=daily_portfolio_table
)
daily_portfolio_value_return_df

date,portfolio_value,daily_return
date,f64,f64
2021-12-31,1.0009e6,0.000913
2022-01-03,985454.386163,-0.015445
2022-01-04,981131.895998,-0.004386
2022-01-05,940208.767557,-0.04171
2022-01-06,937704.99827,-0.002663
…,…,…
2024-12-24,1.9765e6,0.006458
2024-12-26,1.9728e6,-0.001911
2024-12-27,1.9569e6,-0.008037


In [25]:
total_return = calculate_total_return(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    initial_capital=INITIAL_CAPITAL
)
total_return

0.9254577012935494

In [26]:
annualized_return_cagr = calculate_annualized_return_cagr(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    initial_capital=INITIAL_CAPITAL
)
annualized_return_cagr

0.24515245153343113

In [27]:
mean_daily_return = calculate_mean_daily_return(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
mean_daily_return

0.0009971295784436524

In [28]:
annualized_mean_daily_return = calculate_annualized_mean_return(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
annualized_mean_daily_return

0.28550477123025475

## Risk metrics

In [29]:
mean_daily_std = calculate_daily_return_std(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
mean_daily_std

0.015985802023062733

In [30]:
annualized_volatility = calculate_annualized_volatility(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
annualized_volatility

0.25376673996562327

In [31]:
drawdown = calculate_drawdown(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
drawdown

date,portfolio_value,daily_return,drawdown
date,f64,f64,f64
2021-12-31,1.0009e6,0.000913,0.0
2022-01-03,985454.386163,-0.015445,-0.015445
2022-01-04,981131.895998,-0.004386,-0.019763
2022-01-05,940208.767557,-0.04171,-0.060649
2022-01-06,937704.99827,-0.002663,-0.063151
…,…,…,…
2024-12-24,1.9765e6,0.006458,-0.070752
2024-12-26,1.9728e6,-0.001911,-0.072527
2024-12-27,1.9569e6,-0.008037,-0.079981


In [32]:
max_drawdown = calculate_max_drawdown(drawdown_table=drawdown)
max_drawdown

-0.20134209990266316

In [33]:
sharpe_ratio = calculate_sharpe_ratio(
    mean_daily_return=mean_daily_return, annualized_volatility=annualized_volatility, annual_rf=ANNUAL_RISK_FREE_RATE
)
sharpe_ratio

0.8719686977015815

In [34]:
calmer_ratio = calculate_calmer_ratio(
    annualized_return_cagr=annualized_return_cagr, max_drawdown=max_drawdown
)
calmer_ratio

1.2175916097624275

In [35]:
sortino_ratio = calculate_sortino_ratio(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    mean_daily_return=mean_daily_return,
    annual_rf=ANNUAL_RISK_FREE_RATE
)
sortino_ratio

1.2537348745146693

## Turnover metrics

In [36]:
portfolio_turnover_table = get_portfolio_turnover_table(
    rebalance_level_table=rebalance_summary["rebalance_level_table"],
    trade_level_table=rebalance_summary["trade_level_table"],
)
portfolio_turnover_table

rebalance_date,buy_value,sell_value,pre_rebalance_portfolio_value,post_rebalance_portfolio_value,cash_residual,transaction_cost,one_way_turnover,transaction_cost_ratio
date,f64,f64,f64,f64,f64,f64,f64,f64
2021-12-31,997838.877096,0.0,1e6,998976.191123,1137.314027,1023.808877,1.0,0.001024
2022-02-28,854887.898831,855084.116898,856221.430925,854411.858909,-476.039922,1809.572016,0.998443,0.002113
2022-05-02,740541.809826,742557.830461,927211.597011,925607.882371,-63.733927,1603.71464,0.798676,0.00173
2022-06-30,707104.543345,708425.537263,880665.677317,879133.012236,-275.40509,1532.665081,0.802921,0.00174
2022-08-31,797995.859796,800110.557397,999423.805348,997711.873931,127.361094,1711.931417,0.798456,0.001713
…,…,…,…,…,…,…,…,…
2024-04-30,1.6507e6,1.6536e6,1.8367e6,1.8332e6,-906.557785,3454.984999,0.898741,0.001881
2024-07-01,1.5307e6,1.5340e6,1.9034e6,1.9002e6,-777.462812,3225.969912,0.804184,0.001695
2024-09-03,1.9164e6,1.9198e6,1.9190e6,1.9149e6,-1450.29031,4106.055004,0.998616,0.00214


In [37]:
average_turnover = calculate_average_turnover(
    portfolio_turnover_table=portfolio_turnover_table
)
average_turnover

0.7805819674254958

In [38]:
annualized_turnover = calculate_annualized_turnover(
    portfolio_turnover_table=portfolio_turnover_table,
    rebalance_frequency=REBALANCE_FREQUENCY,
    rebalance_freq_unit=REBALANCE_FREQ_UNIT,
)
annualized_turnover

4.6834918045529745

In [39]:
total_transaction_cost = calculate_total_transaction_cost(
    portfolio_turnover_table=portfolio_turnover_table
)
total_transaction_cost

43047.26439354556

In [40]:
total_transaction_cost_to_initial_cap = calculate_total_transaction_cost_ratio(
    portfolio_turnover_table=portfolio_turnover_table, initial_capital=INITIAL_CAPITAL
)
total_transaction_cost_to_initial_cap

0.043047264393545566

In [41]:
average_transaction_cost_ratio = calculate_average_transaction_cost_ratio(
    portfolio_turnover_table=portfolio_turnover_table
)
average_transaction_cost_ratio

0.0016413871560434514

## Benchmark metrics

In [42]:
daily_benchmark_table = get_daily_benchmark_table(
    benchmark_data=benchmark_data,
    benchmark_ticker=BENCHMARK_TICKER,
    backtest_start_date=BACKTEST_START_DATE,
    backtest_end_date=end_date,
)
daily_benchmark_table

date,ticker,open,close,daily_return
date,str,f64,f64,f64
2021-12-31,"""^GSPC""",4775.209961,4766.180176,-0.001891
2022-01-03,"""^GSPC""",4778.140137,4796.560059,0.006374
2022-01-04,"""^GSPC""",4804.509766,4793.540039,-0.00063
2022-01-05,"""^GSPC""",4787.990234,4700.580078,-0.019393
2022-01-06,"""^GSPC""",4693.390137,4696.049805,-0.000964
…,…,…,…,…
2024-12-24,"""^GSPC""",5984.629883,6040.040039,0.011043
2024-12-26,"""^GSPC""",6024.970215,6037.589844,-0.000406
2024-12-27,"""^GSPC""",6006.169922,5970.839844,-0.011056


In [43]:
portfolio_benchmark_returns_table = get_portfolio_benchmark_returns_table(
    daily_benchmark_table=daily_benchmark_table,
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
portfolio_benchmark_returns_table

date,portfolio_return,benchmark_return
date,f64,f64
2021-12-31,0.000913,-0.001891
2022-01-03,-0.015445,0.006374
2022-01-04,-0.004386,-0.00063
2022-01-05,-0.04171,-0.019393
2022-01-06,-0.002663,-0.000964
…,…,…
2024-12-24,0.006458,0.011043
2024-12-26,-0.001911,-0.000406
2024-12-27,-0.008037,-0.011056


In [44]:
portfolio_beta = calculate_portfolio_beta(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table
)
portfolio_beta

0.8871682871367964

In [45]:
jensens_alpha = calculate_jensens_alpha(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table,
    portfolio_beta=portfolio_beta,
    annual_rf=ANNUAL_RISK_FREE_RATE,
)
jensens_alpha

0.17259177375799312

In [46]:
r_squared = calculate_r_squared(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table
)
r_squared

0.3734276273449216

In [47]:
annualized_tracking_error = calculate_annualized_tracking_error(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table
)
annualized_tracking_error

0.20183832623279596

In [48]:
information_ratio = calculate_information_ratio(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table
)
information_ratio

0.8241450046051636

In [49]:
treynor_ratio = calculate_treynor_ratio(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    portfolio_beta=portfolio_beta,
    annual_rf=ANNUAL_RISK_FREE_RATE,
)
treynor_ratio

0.24991438613870542

In [50]:
best_day = get_best_day(daily_portfolio_value_return_df=daily_portfolio_value_return_df)
best_day

date,portfolio_value,daily_return
date,f64,f64
2024-07-31,1.8210e6,0.055243


In [51]:
worst_day = get_worst_day(daily_portfolio_value_return_df=daily_portfolio_value_return_df)
worst_day

date,portfolio_value,daily_return
date,f64,f64
2024-07-24,1.7667e6,-0.064914


In [52]:
win_rate = get_win_rate(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
win_rate

0.5464190981432361